# ML Challenge Overfit et Debordés

## Import Packages

In [129]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor 
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder

## Data Importation

In [130]:
X_train = pd.read_csv("data/challenge_train_features.csv", index_col=0)
y_train = pd.read_csv("data/challenge_train_revenue.csv", index_col=0)
X_test = pd.read_csv("data/challenge_test_features.csv", index_col=0)

Transformations of data using pandas (for date) and for y_train (log transformation)

In [131]:
X_train["date_format"] = pd.to_datetime(X_train["date"], format="%m/%d/%y")
X_train.loc[X_train["date_format"].dt.year > 2025, "date_format"] -= pd.offsets.DateOffset(years=100)
X_train["year"] = X_train["date_format"].dt.year
X_train["month"] = X_train["date_format"].dt.month

X_test["date_format"] = pd.to_datetime(X_test["date"], format="%m/%d/%y")
X_test.loc[X_test["date_format"].dt.year > 2025, "date_format"] -= pd.offsets.DateOffset(years=100)
X_test["year"] = X_test["date_format"].dt.year
X_test["month"] = X_test["date_format"].dt.month

y_train_log = np.log1p(y_train.values)

Transformations of data using sklearn pipeline

In [ ]:

#Transformation functions


def clip_popularity(X):
    X = X.copy()
    X['popularity_score'] = X['popularity_score'].clip(upper=20)
    return X[['popularity_score']]

def budget_missing_indicator(X):
    X= X.copy()
    X['budget_is_zero'] = (X['budget'] == 0).astype(int)
    return X[['budget_is_zero']]

def log_budget(X):
    X = X.copy()
    X['budget'] = np.log1p(X['budget'].clip(lower=0))
    return X[['budget']]

def log_budget_with0(X):
    X= X.copy()
    X['budget_nonzero'] = X['budget'].replace(0, np.nan)
    X['log_budget'] = np.log1p(X['budget_nonzero'])
    return X[['log_budget']]

def collection_to_binary(X):
    return X.notna().astype(int).to_numpy().reshape(-1,1)

def english_to_binary(X):
    return (X == 'en').astype(int).to_numpy().reshape(-1,1)

def US_to_binary(X):
    return (X == 'US').astype(int).to_numpy().reshape(-1,1)


def budget_with_popularity(X):
    X = X.copy()
    X['budget_x_popularity'] = X['budget'] * X['popularity_score']
    return X[['budget_x_popularity']]

def get_first_genre(X):
    X = X.copy()
    X['genre'] = X['genre'].str.split(',').str[0]
    X['genre'] = X['genre'].fillna('Unknown')
    return X[['genre']]

def regrouper_genre(X):
    X = X.copy()
    haut = ["Animation","Adventure", "Family", "Fantasy", "Science Fiction", "Action"]
    
    
    X['grouped_genre'] = X['genre'].apply(lambda g: 'cat1' if g in haut else 'cat2')
    return X[['grouped_genre']]


#pipeline for the variable "genre":
genre_pipe = Pipeline([
    ('get_first', FunctionTransformer(get_first_genre, validate=False)),
    ('regrouper', FunctionTransformer(regrouper_genre, validate=False)),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

### Pipeline


In [133]:
#We create the preprocessing pipeline
#Columns to be transformed
num_cols = ['budget', 'popularity_score']
cat_cols = ['collection', 'language', 'country','month']



preprocessor = ColumnTransformer(
    transformers=[
        # ('budget', 'passthrough', ['budget']),
        # ('budget_missing', FunctionTransformer(budget_missing_indicator, validate=False), ['budget']),
        ('log_budget', FunctionTransformer(log_budget_with0, validate=False), ['budget']),
        # ('budget_pop', FunctionTransformer(budget_with_popularity, validate=False), ['budget', 'popularity_score']),
        ('popularity','passthrough', ['popularity_score']),
        ('length','passthrough', ['length']),
        ('collection_bin', FunctionTransformer(collection_to_binary, validate=False), ['collection']),
        ('language_bin', FunctionTransformer(english_to_binary, validate=False), ['language']),
        ('country_bin', FunctionTransformer(US_to_binary, validate=False), ['country']),
        ('month_cat', OneHotEncoder(handle_unknown='ignore'), ['month']),
        ('genre_cat', genre_pipe, ['genre'])
    ],
    remainder='drop'
)


pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

#We apply the pipeline to the training dataset
X_train_transformed = pipeline.fit_transform(X_train)

#We apply the same pipeline to the test dataset
X_test_transformed = pipeline.transform(X_test)


Model

In [134]:
model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    min_child_weight=1.0,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_transformed, y_train_log)

y_pred_log= model.predict(X_test_transformed)

y_test_pred = np.expm1(y_pred_log).clip(0, None)


Saving in a text file

In [135]:
pred_str = ",".join([str(int(p)) for p in y_test_pred])  

with open("xgboost.txt", "w") as f:
    f.write(pred_str)

In [136]:
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_squared_log_error

# Catégories : ici aucune car toutes sont numériques (0/1)
train_pool = Pool(X_train_transformed, y_train_log)

model = CatBoostRegressor(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',  # on apprend sur log(y)
    random_seed=2,
    verbose=100
)

model.fit(train_pool)

# Prédictions
y_pred_log = model.predict(X_test_transformed)
y_pred = np.expm1(y_pred_log) 

0:	learn: 2.8520498	total: 2.86ms	remaining: 2.28s
100:	learn: 1.9017983	total: 241ms	remaining: 1.67s
200:	learn: 1.7423869	total: 511ms	remaining: 1.52s
300:	learn: 1.6279679	total: 748ms	remaining: 1.24s
400:	learn: 1.5243838	total: 981ms	remaining: 977ms
500:	learn: 1.4475252	total: 1.21s	remaining: 721ms
600:	learn: 1.3755260	total: 1.44s	remaining: 476ms
700:	learn: 1.3133512	total: 1.68s	remaining: 238ms
799:	learn: 1.2622419	total: 1.91s	remaining: 0us


In [137]:
pred_str = ",".join([str(int(p)) for p in y_pred])

with open("catboost.txt", "w") as f:
    f.write(pred_str)

y_pred = 0